In [1]:
import pandas as pd, sqlite3
con = sqlite3.connect("../data/clean/posts.db")

con.executescript("""
DROP TABLE IF EXISTS posts;
CREATE TABLE posts (
    video_id TEXT PRIMARY KEY, title TEXT, channel_title TEXT,
    category TEXT, content_type TEXT, publish_time TEXT,
    post_date TEXT, post_day TEXT, post_hour INTEGER,
    views INTEGER, likes INTEGER, comment_count INTEGER,
    engagement_rate REAL,
    is_deleted INTEGER, is_zero_views INTEGER, is_outlier INTEGER
);
""")

df = pd.read_csv("../data/clean/posts_clean.csv")
for c in ["is_deleted", "is_zero_views", "is_outlier"]:
    df[c] = df[c].astype(int)
df.to_sql("posts", con, if_exists="append", index=False)
print(pd.read_sql("SELECT COUNT(*) AS rows_loaded FROM posts", con))

   rows_loaded
0         2073


In [2]:
con.executescript(open("../sql/03_views.sql").read())

In [3]:
pd.read_sql("""
SELECT video_id, title, views, likes, comment_count,
       ROUND((likes + comment_count) * 100.0 / views, 4) AS er
FROM v_valid_posts
ORDER BY er DESC
LIMIT 10
""", con)

,video_id,title,views,likes,comment_count,er
0,Vm4Xn5vjqRA,"CRISTIANO RONALDO E FRED, O GRANDE ENCONTRO",5938439,1121217,83660,20.2895
1,dH4ZXmeEJXs,STRANGERS DO MY MAKEUP ON HOLLYWOOD BOULEVARD,2128508,264835,136746,18.8668
2,ONI_06wGbsQ,THINGS THAT ARE LOWER,180546,29537,2409,17.6941
3,xf-dywd3zx0,Official Trailer | BTS: Burn The Stage,3132248,490749,34948,16.7834
4,xPsEiZAOb_I,GIVING AWAY MY YOUTUBE ADSENSE,296755,41687,2897,15.0238
5,pDi7cezk9CA,COMING OUT (ELLE MILLS STYLE),1275330,168189,22931,14.9859
6,7Bgj9c48zHU,How to do a beauty Vlog for Tarte Cosmetics,119519,13367,4295,14.7776
7,HDDIWGlBt3o,My Gender.,553866,66026,8479,13.4518
8,QS8bma7LRX4,Natalie's 2nd Rap - SNL,39720,4800,530,13.4189
9,U5EDORs8Jkk,Gateway to Sedona DEBUNK,380760,45444,3180,12.7702


In [4]:
pd.read_sql("SELECT * FROM v_er_by_day ORDER BY avg_er DESC", con)

,post_day,videos,avg_er
0,Saturday,208,3.044
1,Thursday,337,2.695
2,Wednesday,371,2.635
3,Friday,274,2.562
4,Sunday,203,2.559
5,Tuesday,343,2.497
6,Monday,335,2.396


In [5]:
pd.read_sql("""
SELECT * FROM v_er_by_hour
WHERE videos >= 30
ORDER BY avg_er DESC
""", con)

,post_hour,videos,avg_er
0,1,108,3.773
1,0,112,3.553
2,5,59,3.382
3,2,95,3.348
4,3,89,3.310
5,4,64,2.986
6,23,125,2.953
7,22,164,2.939
8,17,52,2.698
9,20,148,2.687


In [6]:
pd.read_sql("SELECT * FROM v_er_by_type ORDER BY avg_er DESC", con)


,content_type,videos,avg_er
0,Behind the Scenes / Vlog,23,6.199
1,Food / Lifestyle,29,3.571
2,Challenge / Stunt / Comedy,46,3.204
3,Other,847,3.183
4,Music,164,2.437
5,Trailer / Promo,281,2.280
6,Interview / Talk Show,229,2.096
7,Talent / Competition Show,57,1.973
8,Sports Debate / Analysis,113,1.856
9,Sports Highlights,282,1.659


In [7]:
pd.read_sql("""
SELECT category, COUNT(*) AS videos, ROUND(AVG(engagement_rate), 3) AS avg_er
FROM v_valid_posts
GROUP BY category
""", con)

,category,videos,avg_er
0,Entertainment,1620,2.894
1,Sports,451,1.576


In [8]:
pd.read_sql("SELECT * FROM v_er_by_day", con).to_csv("../data/clean/sql_by_day.csv", index=False)
pd.read_sql("SELECT * FROM v_er_by_hour", con).to_csv("../data/clean/sql_by_hour.csv", index=False)
pd.read_sql("SELECT * FROM v_er_by_type", con).to_csv("../data/clean/sql_by_type.csv", index=False)
print("Saved")

Saved


In [9]:
pd.read_sql("""
SELECT COUNT(*) AS videos,
       ROUND(AVG(engagement_rate), 3) AS avg_er,
       ROUND(MAX(engagement_rate), 2) AS max_er
FROM v_valid_posts
""", con)

,videos,avg_er,max_er
0,2071,2.607,20.29


In [10]:
pd.read_sql("SELECT COUNT(*) FROM v_valid_posts", con)

,COUNT(*)
0,2071
